In [1]:
import ipdb # <- трасировка и точки останова
import header
from header import __root__

# Internal modules
from src import gs

🔑 Found password in password.txt (DEBUG MODE)
✅ Successfully opened KeePass database: C:\Users\user\Documents\repos\hypotez\secrets\credentials.kdbx
Failed to load GAPI credentials


In [2]:
import importlib
import os
import asyncio
import time
from pathlib import Path
from types import SimpleNamespace
from typing import Optional, List, Any
from dataclasses import dataclass, field


from src.suppliers.suppliers_list import *
from src.suppliers.get_graber_by_supplier  import get_graber_by_supplier_prefix, get_graber_by_supplier_url
from src.suppliers.graber import Graber
from src.webdriver.driver import Driver
from src.webdriver.firefox import Firefox
from src.webdriver.chrome import Chrome
from src.llm.gemini import GoogleGenerativeAi
from src.llm.openai.model import OpenAIModel
from src.endpoints.prestashop.product import PrestaProduct
from src.endpoints.prestashop.language import PrestaLanguage
from src.endpoints.prestashop.product_fields import ProductFields
from src.endpoints.advertisement.facebook.scenarios.post_message import (
    post_message,
)
from src.utils.file import read_text_file, save_text_file, get_filenames_from_directory

from src.utils.jjson import j_loads, j_loads_ns, j_dumps
from src.utils.image import get_image_bytes, get_raw_image_data
from src.utils.printer import pprint as print
from src.logger.logger import logger

2025-05-25 12:15:45,853 - INFO - Anonymized telemetry enabled. See https://docs.browser-use.com/development/telemetry for more information.


In [3]:
# --- file config.py
class Config:
    ENDPOINT: Path = __root__ /'SANDBOX' / 'davidka'
    SUPPLIERS_ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list'
    config:SimpleNamespace = j_loads_ns(ENDPOINT / 'davidka.json')
    GEMINI_API_KEY:str = gs.credentials.gemini.onela.api_key
    PRESTA_API_KEY:str = gs.credentials.prestashop.store_davidka_net.api_key
    PRESTA_DOMAIN:str = gs.credentials.prestashop.store_davidka_net.api_domain
    gemini_model_name:str = config.gemini_model_name
    system_instruction:str = ' ' # <- Это пробел!
    webdriver_window_mode:str = 'headless'
# --- end file config.pt

In [4]:

async def description_short(self, value:Optional[str] = '') -> bool:
    """Fetch and set short description.
    
    Args:
    value (atr): это значение можно передать в словаре kwargs через ключ {description_short = `value`} при определении класса.
    Если `value` было передано, его значение подставляется в поле `ProductFields.description_short`.
    """

    try:
        # Получаем значение через execute_locator
        value =  value or await self.driver.execute_locator(self.product_locator.description_short)
        if not value:
            ...
            return
        self.fields.description_short = normalize_string()
        return True

    except Exception as ex:
        logger.error(f"Ошибка получения значения в поле `description_short`", ex)
        ...
        return

    self.fields.description_short = value
    return True

In [5]:
async def get_list_products_in_category (d: Driver, l: SimpleNamespace) -> list:    
    """ 
    Функция извлекает список URL-адресов товаров со страницы категории.
    При необходимости пролистывает страницы категорий.

    Args:
        d (Driver): Экземпляр WebDriver.
        l (SimpleNamespace): Объект с локаторами для страницы категории, 
                             включая локаторы товаров и пагинации.
    
    Returns:
        List[str] | None: Список URL-адресов товаров или `None`, если товары не найдены.
    
    Example:
        >>> # Пример использования (требует настройки d и l)
        >>> # driver = Driver(...) 
        >>> # locators = SimpleNamespace(product_links=..., pagination_locators=...)
        >>> # product_urls = await get_list_products_in_category(driver, locators)
        >>> # if product_urls:
        >>> #     print(f'Найдено {len(product_urls)} товаров.')
    """


    """
       В текущей версии пагинация происхоадит через нажати кнопки
       https://hbdeadsea.co.il/collections/<название каетегории>?page=...
    """

    """
    all_product_urls: List[str] = []
    # Извлечение ссылок на товары с текущей (первой) страницы
    while True:
        if not await d.execute_locator(l.show_more):
            break
        product_links: List[str] | str | None = await d.execute_locator(l.product_links)
        if len(all_product_urls) <  len(product_links):
            all_product_urls.extend(all_product_urls)
            time.sleep(3)
            print('Листаю')
            continue
        else:
            break

    """
    product_links: List[str] | str | None = await d.execute_locator(l.product_links)
    return product_links if isinstance(product_links, list) else [product_links]


class Scenario:
    """Dataclass for designing and promoting images through various platforms."""

    gemini: Optional[GoogleGenerativeAi] = None
    openai: Optional[OpenAIModel] = None
    product: PrestaProduct = None
    driver: Driver = None
    graber: Graber = None

    def __init__(self,
            presta_api_key:Optional[str] = '',
            presta_api_domain:Optional[str] = '',
            gemini_model_name:Optional[str] = '',
            openai_model_name:Optional[str] = '',
            gemini_api_key:Optional[str] = '',
            openai_api_key:Optional[str] = '',
            gemini: Optional[GoogleGenerativeAi] = None, 
            openai: Optional[OpenAIModel] = None,
            system_instruction:str = '',
            driver:Driver = None, 
            webdriver_window_mode:str = ''
            ):
        """
        Инициализация 
            Args:
                presta_api_key:Optional[str] = '',
                presta_api_domain:Optional[str] = '',
                gemini_model_name:Optional[str] = '',
                openai_model_name:Optional[str] = '',
                gemini_api_key:Optional[str] = '',
                openai_api_key:Optional[str] = '',
                gemini: Optional[GoogleGenerativeAi] = None, 
                openai: Optional[OpenAIModel] = None,
        """
        ...
        if driver:
            self.driver = driver
        else:
            self.driver = Driver(Firefox, 
                                 window_mode=webdriver_window_mode if webdriver_window_mode else Config.webdriver_window_mode,
                                )

        if gemini:
            self.gemini = gemini
        else:
            gemini_api_key:str = gemini_api_key if gemini_api_key else Config.GEMINI_API_KEY
            gemini_model_name:str = gemini_model_name if gemini_model_name else Config.gemini_model_name
            system_instruction:str = system_instruction if system_instruction else Config.system_instruction
            if not self._init_gemini(gemini_api_key, gemini_model_name, system_instruction):
                logger.debug('Модель GEMINI не иницаилизирована')
                

        presta_api_key:str = presta_api_key if presta_api_key else Config.PRESTA_API_KEY
        presta_api_domain:str = presta_api_domain if presta_api_domain else Config.PRESTA_DOMAIN
        if not presta_api_key or not presta_api_domain:
            logger.critical(f'Проверь \nAPI {presta_api_key}\nDomain {presta_api_domain=}')
            return False

        self.product = PrestaProduct(presta_api_key, presta_api_domain )

    def _init_gemini(self, api_key: str, model_name: str, system_instruction: str) -> bool:
        """"""
        try:
            generation_config = dict({'response_mime_type':'application/json'})
            self.gemini = GoogleGenerativeAi(api_key, model_name, generation_config, system_instruction)
            return True
        except Exception as ex:
            logger.error(f'Ошибка иницализации модели!', ex, False)
            return False


    async def process_supplier(self, supplier_prefix:str) -> bool:
        """"""
        ...
        try:
            supplier_path:Path = Config.SUPPLIERS_ENDPOINT / supplier_prefix 
            self.graber = get_graber_by_supplier_prefix(self.driver, supplier_prefix)
            scenarios_list: list[dict] = j_loads(Config.SUPPLIERS_ENDPOINT / supplier_prefix / 'scenarios')
            locators_path:Path = supplier_path / 'locators' 
            locator_product:SimpleNamespace = j_loads_ns(locators_path / 'product.json')
            locator_category:SimpleNamespace = j_loads_ns(locators_path / 'category.json')
            categories_crawler:Any = None
            categories_crawler_module_path:str = f"src.suppliers.suppliers_list.{supplier_prefix}.categories_crawler"
        except Exception as ex:
            logger.error(f'Непредвиденная ошибка', ex)
            return False

        try:
            categories_crawler = importlib.import_module(categories_crawler_module_path)
        except Exception as ex:
            logger.error(f"Failed to import module `categories_crawler` '{supplier_prefix}'", ex)
            return False
        
        for scenario in scenarios_list:
            for _, item in scenario.items():
                self.driver.get_url(item['url'])
    
                products_urls_in_category:list = await get_list_products_in_category(self.driver, locator_category)
                if not products_urls_in_category:

                    continue # <- мб пустаая категория
                for product_url in products_urls_in_category:
                    self.driver.get_url(product_url)

                    # Не все поля товара надо заполнять. Вот кортеж необходимых полей:
                    required_fields:tuple = ('id_manufacturer',
                                        'id_supplier',
                                        'name',                                                
                                        'description',
                                        'description_short',
                                        'default_image_url',
                                        )
                    self.graber.description_short = description_short
                    product_fields:ProductFields = await self.graber.grab_page_async(*required_fields)
                    ipdb.set_trace()
                ...

            
    async def process_suppliers_list(self, suppliers_prefixes: str|list) -> bool:
        """
        Process suppliers based on the provided prefix.
        Args:
            suppliers_prefixes (Optional[str | List[str, str]], optional): Prefix for suppliers. Defaults to ''.
        Returns:
            bool: True if processing is successful, False otherwise.
        Raises:
            Exception: If any error occurs during supplier processing.
        """
        
        for supplier_prefix in suppliers_prefixes:
            try:
                await self.process_supplier(supplier_prefix)
            except Exception as ex:
                logger.error(f'Error while processing suppliers: {ex}')
                continue

In [6]:
driver:Driver = None

try:
    driver = Driver(Firefox, window_mode = 'normal')
except Exception as ex:
    logger.critical(f'Ошибка инициализации шебдрайвера: ', ех, False)  

2025-05-25 12:15:46,125 - INFO - ℹ️ Инициализация Firefox WebDriver 
2025-05-25 12:15:46,125 - DEBUG - 🐛 Текущий __root__: C:\Users\user\Documents\repos\hypotez 
NoneType: None
2025-05-25 12:15:46,131 - DEBUG - 🐛 Конфигурация загружена. enable_geckodriver_log: True 
NoneType: None
2025-05-25 12:15:46,131 - INFO - ℹ️ Попытка настроить логирование geckodriver... 
2025-05-25 12:15:46,133 - DEBUG - 🐛 Предполагаемый путь к лог-файлу geckodriver: C:\Users\user\Documents\repos\hypotez\geckodriver.log 
NoneType: None
2025-05-25 12:15:46,135 - INFO - ℹ️ Логирование geckodriver настроено. Путь к лог-файлу: C:\Users\user\Documents\repos\hypotez\geckodriver.log 
2025-05-25 12:15:51,615 - INFO - ℹ️ Браузер Firefox успешно запущен. Режим окна: normal. 


In [ ]:

scenario: Scenario = Scenario(driver = driver)
suppliers_prefixes_list:list = ['hb']  
products_urls_in_category:list = [] 
product_fields:ProductFields = None

await scenario.process_suppliers_list(suppliers_prefixes_list)

    

2025-05-25 12:15:51,624 - INFO - ℹ️ Модель models/gemini-2.5-flash-preview-04-17 инициализирована 
2025-05-25 12:15:51,925 - ERROR - ❌ Нет соединения. response.reason='Unauthorized' 
NoneType: None
2025-05-25 12:15:51,930 - INFO - ℹ️ Переход на URL: https://hbdeadsea.co.il/collections/%D7%98%D7%99%D7%A4%D7%95%D7%97-%D7%94%D7%A4%D7%A0%D7%99%D7%9D 
2025-05-25 12:15:54,663 - DEBUG - 🐛 Попытка 1/10: readyState=complete для https://hbdeadsea.co.il/collections/%D7%98%D7%99%D7%A4%D7%95%D7%97-%D7%94%D7%A4%D7%A0%D7%99%D7%9D 
NoneType: None
2025-05-25 12:15:54,665 - INFO - ℹ️ Страница загружена (readyState=complete): https://hbdeadsea.co.il/collections/%D7%98%D7%99%D7%A4%D7%95%D7%97-%D7%94%D7%A4%D7%A0%D7%99%D7%9D 
2025-05-25 12:15:54,668 - INFO - ℹ️ Фактический URL после перехода: https://hbdeadsea.co.il/collections/%D7%98%D7%99%D7%A4%D7%95%D7%97-%D7%94%D7%A4%D7%A0%D7%99%D7%9D 
2025-05-25 12:15:54,669 - DEBUG - 🐛 Предыдущий URL сохранен:  
NoneType: None
2025-05-25 12:15:54,688 - INFO - ℹ️ Перех

> c:\users\user\appdata\local\temp\ipykernel_34952\464379219.py(150)process_supplier()

